# Stage 4: Official Code Correspondence Audit

This notebook instruments the *official* SOAP optimizer implementation (adapted from `official_soap.py`) with our MoQ logic and layer diagnostics. We are testing whether the official implementation exhibits the same catastrophic staleness amplification as our mathematically exact paper reconstruction.

In [ ]:
import json
import math
from pathlib import Path
from typing import Dict, List, Optional
from itertools import chain

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
class InstrumentedOfficialSOAP(optim.Optimizer):
    """
    Official SOAP implementation instrumented with MoQ and audit diagnostics.
    """
    def __init__(
        self,
        params,
        lr: float = 1e-3,
        betas=(0.95, 0.95),
        shampoo_beta: float= -1,
        eps: float = 1e-8,
        weight_decay: float = 0.01,
        precondition_frequency: int=10,
        max_precond_dim: int=10000,
        merge_dims: bool = False,
        precondition_1d: bool = False,
        normalize_grads: bool = False,
        correct_bias: bool = True,
        mu: float = 0.0,
    ):
        defaults = {
            "lr": lr,
            "betas": betas,
            "shampoo_beta": shampoo_beta,
            "eps": eps,
            "weight_decay": weight_decay,
            "precondition_frequency": precondition_frequency,
            "max_precond_dim": max_precond_dim,
            "merge_dims": merge_dims,
            "precondition_1d": precondition_1d,
            "normalize_grads": normalize_grads,
            "correct_bias": correct_bias,
            "mu": mu,
        }
        super().__init__(params, defaults)
        self._data_format = "channels_first"
        self.layer_diagnostics = []
        
    def merge_dims(self, grad, max_precond_dim):
        shape = grad.shape
        new_shape = []
        curr_shape = 1
        for sh in shape:
            temp_shape = curr_shape * sh
            if temp_shape > max_precond_dim:
                if curr_shape > 1:
                    new_shape.append(curr_shape)
                    curr_shape = sh
                else:
                    new_shape.append(sh)
                    curr_shape = 1
            else:
                curr_shape = temp_shape
        if curr_shape > 1 or len(new_shape)==0:
            new_shape.append(curr_shape)
        return grad.reshape(new_shape)               

    @torch.no_grad()
    def step(self, closure = None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        for group in self.param_groups:
            for layer_idx, p in enumerate(group["params"]):
                if p.grad is None:
                    continue
                
                raw_gradient = p.grad
                state = self.state[p]
                
                # MoQ INJECTION
                if "prev_moq_grad" not in state:
                    state["prev_moq_grad"] = torch.zeros_like(raw_gradient)
                
                step_idx = state.get("step", 0) + 1
                mu = group["mu"]
                
                if step_idx == 1 or mu == 0.0:
                    grad = raw_gradient.clone()
                else:
                    grad = (1 + mu) * raw_gradient - mu * state["prev_moq_grad"]
                state["prev_moq_grad"].copy_(raw_gradient)
                
                # BEGIN OFFICIAL SOAP LOGIC USING `grad`
                if "step" not in state:
                    state["step"] = 0 
                    
                if "exp_avg" not in state:
                    state["exp_avg"] = torch.zeros_like(grad)
                    state["exp_avg_sq"] = torch.zeros_like(grad)
                
                if 'Q' not in state:
                    self.init_preconditioner(
                        grad, state,
                        precondition_frequency=group['precondition_frequency'],
                        precondition_1d=group['precondition_1d'],
                        shampoo_beta=(group['shampoo_beta'] if group['shampoo_beta'] >= 0 else group["betas"][1]),
                        max_precond_dim=group['max_precond_dim'],
                        merge_dims=group["merge_dims"],
                    )
                    self.update_preconditioner(grad, state,
                                               max_precond_dim=group['max_precond_dim'],
                                               merge_dims=group["merge_dims"],
                                               precondition_1d=group["precondition_1d"])
                    
                    # Log skipped step diagnostic
                    self.layer_diagnostics.append({
                        "layer": layer_idx,
                        "raw_grad_norm": float(torch.linalg.vector_norm(raw_gradient)),
                        "MoQ_raw_ratio": float(torch.linalg.vector_norm(grad)) / (float(torch.linalg.vector_norm(raw_gradient)) + 1e-12) if mu > 0.0 else 1.0,
                        "precond_norm": 0.0,
                        "precond_gain": 0.0,
                        "skipped": True
                    })
                    continue
                
                grad_projected = self.project(grad, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])

                exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                beta1, beta2 = group["betas"]
                state["step"] += 1

                exp_avg.mul_(beta1).add_(grad_projected, alpha=(1.0 - beta1))
                exp_avg_sq.mul_(beta2).add_(grad_projected.square(), alpha=(1.0 - beta2))

                denom = exp_avg_sq.sqrt().add_(group["eps"])
                exp_avg_projected = exp_avg
                
                step_size = group["lr"]
                if group["correct_bias"]:
                    bias_correction1 = 1.0 - beta1 ** state["step"]
                    bias_correction2 = 1.0 - beta2 ** state["step"]
                    step_size = step_size * (bias_correction2 ** .5) / bias_correction1

                norm_grad = self.project_back(exp_avg_projected / denom, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])

                if group["normalize_grads"]:
                    norm_grad = norm_grad / (1e-30+torch.mean(norm_grad**2)**0.5)
                
                p.add_(norm_grad, alpha=-step_size)
                
                if group["weight_decay"] > 0.0:
                    p.add_(p, alpha=(-group["lr"] * group["weight_decay"]))
                    
                self.update_preconditioner(grad, state, max_precond_dim=group['max_precond_dim'], merge_dims=group["merge_dims"], precondition_1d=group["precondition_1d"])
                
                # DIAGNOSTICS FOR AUDIT
                raw_norm = float(torch.linalg.vector_norm(raw_gradient))
                moq_norm = float(torch.linalg.vector_norm(grad))
                update_norm = float(torch.linalg.vector_norm(norm_grad))
                
                self.layer_diagnostics.append({
                    "layer": layer_idx,
                    "raw_grad_norm": raw_norm,
                    "MoQ_raw_ratio": (moq_norm / (raw_norm + 1e-12)) if mu > 0.0 else 1.0,
                    "precond_norm": update_norm,
                    "precond_gain": update_norm / (moq_norm + 1e-12),
                    "skipped": False
                })
        return loss
    
    def init_preconditioner(self, grad, state, precondition_frequency=10, shampoo_beta=0.95, max_precond_dim=10000, precondition_1d=False, merge_dims=False):
        state['GG'] = []
        if grad.dim() == 1:
            if not precondition_1d or grad.shape[0] > max_precond_dim:
                state['GG'].append([])
            else:
                state['GG'].append(torch.zeros(grad.shape[0], grad.shape[0], device=grad.device))
        else:
            if merge_dims:
                grad = self.merge_dims(grad, max_precond_dim)
            for sh in grad.shape:
                if sh > max_precond_dim:
                    state['GG'].append([])
                else:
                    state['GG'].append(torch.zeros(sh, sh, device=grad.device))
        state['Q'] = None
        state['precondition_frequency'] = precondition_frequency
        state['shampoo_beta'] = shampoo_beta          
        
    def project(self, grad, state, merge_dims=False, max_precond_dim=10000):
        original_shape = grad.shape
        if merge_dims:
            grad = self.merge_dims(grad, max_precond_dim)
        for mat in state['Q']:
            if len(mat) > 0:
                grad = torch.tensordot(grad, mat, dims=[[0], [0]])
            else:
                permute_order = list(range(1, len(grad.shape))) + [0]
                grad = grad.permute(permute_order)
        if merge_dims:
            grad = grad.reshape(original_shape)
        return grad
        
    def update_preconditioner(self, grad, state, max_precond_dim=10000, merge_dims=False, precondition_1d=False):
        if state["Q"] is not None:
            state["exp_avg"] = self.project_back(state["exp_avg"], state, merge_dims=merge_dims, max_precond_dim=max_precond_dim)
        if grad.dim() == 1:
            if precondition_1d and grad.shape[0] <= max_precond_dim:
                state['GG'][0].lerp_(grad.unsqueeze(1) @ grad.unsqueeze(0), 1-state['shampoo_beta'])
        else:
            if merge_dims:
                new_grad = self.merge_dims(grad, max_precond_dim)
                for idx, sh in enumerate(new_grad.shape):
                    if sh <= max_precond_dim:
                        outer_product = torch.tensordot(new_grad, new_grad, dims=[[*chain(range(idx), range(idx + 1, len(new_grad.shape)))]] * 2)
                        state['GG'][idx].lerp_(outer_product, 1-state['shampoo_beta'])
            else:
                for idx, sh in enumerate(grad.shape):
                    if sh <= max_precond_dim:
                        outer_product = torch.tensordot(grad, grad, dims=[[*chain(range(idx), range(idx + 1, len(grad.shape)))]] * 2)
                        state['GG'][idx].lerp_(outer_product, 1-state['shampoo_beta'])
                     
        if state['Q'] is None:
            state['Q'] = self.get_orthogonal_matrix(state['GG'])
        if state['step'] > 0 and state['step'] % state['precondition_frequency'] == 0:
            state['Q'] = self.get_orthogonal_matrix_QR(state, max_precond_dim, merge_dims)

        if state["step"] > 0:
            state["exp_avg"] = self.project(state["exp_avg"], state, merge_dims=merge_dims, max_precond_dim=max_precond_dim) 

    def project_back(self, grad, state, merge_dims=False, max_precond_dim=10000):
        original_shape = grad.shape
        if merge_dims:
            grad = self.merge_dims(grad, max_precond_dim)
        for mat in state['Q']:
            if len(mat) > 0:
                grad = torch.tensordot(grad, mat, dims=[[0], [1]])
            else:
                permute_order = list(range(1, len(grad.shape))) + [0]
                grad = grad.permute(permute_order)
        if merge_dims:
            grad = grad.reshape(original_shape)
        return grad
        
    def get_orthogonal_matrix(self, mat):
        matrix = []
        for m in mat:
            if len(m) == 0:
                matrix.append([])
                continue
            if m.data.dtype != torch.float:
                float_data = False
                original_type = m.data.dtype
                original_device = m.data.device
                matrix.append(m.data.float())
            else:
                float_data = True
                matrix.append(m.data)
        
        final = []
        for m in matrix:
            if len(m) == 0:
                final.append([])
                continue
            try:
                _, Q = torch.linalg.eigh(m+1e-30*torch.eye(m.shape[0], device=m.device))
            except:
                _, Q = torch.linalg.eigh(m.to(torch.float64)+1e-30*torch.eye(m.shape[0], device=m.device))
                Q = Q.to(m.dtype)
            Q = torch.flip(Q, [1])
            if not float_data:
                Q = Q.to(original_device).type(original_type)
            final.append(Q)
        return final
        
    def get_orthogonal_matrix_QR(self, state, max_precond_dim=10000, merge_dims=False):
        precond_list = state['GG']
        orth_list = state['Q']
        matrix = []
        orth_matrix = []
        for m,o in zip(precond_list, orth_list):
            if len(m) == 0:
                matrix.append([])
                orth_matrix.append([])
                continue
            if m.data.dtype != torch.float:
                float_data = False
                original_type = m.data.dtype
                original_device = m.data.device
                matrix.append(m.data.float())
                orth_matrix.append(o.data.float())
            else:
                float_data = True
                matrix.append(m.data.float())
                orth_matrix.append(o.data.float())
        
        orig_shape = state['exp_avg_sq'].shape
        if merge_dims:
            exp_avg_sq = self.merge_dims(state['exp_avg_sq'], max_precond_dim)
        else:
            exp_avg_sq = state['exp_avg_sq']
            
        final = []
        for ind, (m,o) in enumerate(zip(matrix, orth_matrix)):
            if len(m)==0:
                final.append([])
                continue
            est_eig = torch.diag(o.T @ m @ o)
            sort_idx = torch.argsort(est_eig, descending=True)
            exp_avg_sq = exp_avg_sq.index_select(ind, sort_idx)
            o = o[:,sort_idx]
            power_iter = m @ o
            Q, _ = torch.linalg.qr(power_iter)
            if not float_data:
                Q = Q.to(original_device).type(original_type)
            final.append(Q)
        
        if merge_dims:
            exp_avg_sq = exp_avg_sq.reshape(orig_shape)
                
        state['exp_avg_sq'] = exp_avg_sq
        return final


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_official_audit(K: int, steps: int, use_moq: bool, mu: float = 0.85, seed: int = 0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    lr = 1e-3
    opt_mu = mu if use_moq else 0.0
    opt = InstrumentedOfficialSOAP(model.parameters(), lr=lr, betas=(0.95, 0.95), precondition_frequency=10, mu=opt_mu)

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        err = relative_l2(model, device)
        
        step_data = {
            "step": step,
            "loss": float(loss.detach()),
            "rel_l2": float(err),
            "layers": opt.layer_diagnostics
        }
        history.append(step_data)
        
        print(f"Step {step} | Loss {loss.item():.4e} | RelL2 {err:.4e}")

    return history


### Run Stage 4 Audit (20 steps each, using Official SOAP Code Logic)

In [ ]:
print("Running K=1 Official SOAP...")
audit_K1_soap = run_official_audit(K=1, steps=20, use_moq=False)
with open("stage4_official_audit_K1_soap.json", "w") as f:
    json.dump(audit_K1_soap, f, indent=2)


In [ ]:
print("\nRunning K=1 Official SOAP+MoQ...")
audit_K1_moq = run_official_audit(K=1, steps=20, use_moq=True)
with open("stage4_official_audit_K1_moq.json", "w") as f:
    json.dump(audit_K1_moq, f, indent=2)


In [ ]:
print("\nRunning K=20 Official SOAP...")
audit_K20_soap = run_official_audit(K=20, steps=20, use_moq=False)
with open("stage4_official_audit_K20_soap.json", "w") as f:
    json.dump(audit_K20_soap, f, indent=2)


In [ ]:
print("\nRunning K=20 Official SOAP+MoQ...")
audit_K20_moq = run_official_audit(K=20, steps=20, use_moq=True)
with open("stage4_official_audit_K20_moq.json", "w") as f:
    json.dump(audit_K20_moq, f, indent=2)
